In [35]:
%load_ext autoreload
%autoreload 2

import io
import pandas as pd
import httpx
import base64
import datetime

from upath import UPath

import wave_buoys_viewer as wbv

data_dir = UPath("../data")
data_path = data_dir / "wave_buoys_data_2_years.parquet"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# Scrap wave buoy data
campaign_id = "camp=06403"
data = wbv.scrap.scrap_wave_buoy_data(campaign_id=campaign_id)

# Create directory if it does not exist
data_path.parent.mkdir(parents=True, exist_ok=True)

# Save data to parquet file
data.to_parquet(str(data_path), index=False)

data

,campaign_id,datetime,height_1_3_m,height_max_m,period_1_3_s,peak_direction_deg,peak_spread_deg,sea_temperature_c
0,camp=06403,2025-10-19 19:30:00,0.5,0.9,9.0,298,23,18.7
1,camp=06403,2025-10-19 19:00:00,0.5,0.9,8.8,304,19,18.8
2,camp=06403,2025-10-19 18:30:00,0.5,0.9,9.1,305,21,18.8
3,camp=06403,2025-10-19 18:00:00,0.5,0.9,8.8,311,28,18.9
4,camp=06403,2025-10-19 17:30:00,0.4,0.9,7.8,298,29,18.8
...,...,...,...,...,...,...,...,...
92,camp=06403,2025-10-17 21:30:00,0.3,0.6,4.5,308,24,19.5
93,camp=06403,2025-10-17 21:00:00,0.3,0.6,4.7,297,27,19.5
94,camp=06403,2025-10-17 20:30:00,0.3,0.6,5.0,305,22,19.6
95,camp=06403,2025-10-17 20:00:00,0.3,0.5,5.1,294,20,19.7


In [37]:
# Set date range
start_date = datetime.datetime.strptime("2023-01-01", "%Y-%m-%d")
end_date = datetime.datetime.strptime("2025-11-30", "%Y-%m-%d")

# Delta to repeat
delta_to_repeat = data["datetime"].max() - data["datetime"].min()
num_repeats = (end_date - start_date) // delta_to_repeat + 1

# Repeat data to cover the date range
all_data = []
for i in range(num_repeats):
    delta = i * delta_to_repeat
    df_copy = data.copy()
    df_copy["datetime"] = df_copy["datetime"] - delta
    all_data.append(df_copy)

# Concatenate all data and filter by date range
data = pd.concat(all_data, ignore_index=True)
data = data[(data["datetime"] >= start_date) & (data["datetime"] <= end_date)]

# Move datetime to the second column
data = data[["campaign_id", "datetime"] + [col for col in data.columns if col not in ["campaign_id", "datetime"]]]

# Sort data by datetime
data = data.sort_values(by="datetime", ascending=False)

# Reset index
data = data.reset_index(drop=True)

data.to_parquet(str(data_path), index=False)

data

,campaign_id,datetime,height_1_3_m,height_max_m,period_1_3_s,peak_direction_deg,peak_spread_deg,sea_temperature_c
0,camp=06403,2025-10-19 19:30:00,0.5,0.9,9.0,298,23,18.7
1,camp=06403,2025-10-19 19:00:00,0.5,0.9,8.8,304,19,18.8
2,camp=06403,2025-10-19 18:30:00,0.5,0.9,9.1,305,21,18.8
3,camp=06403,2025-10-19 18:00:00,0.5,0.9,8.8,311,28,18.9
4,camp=06403,2025-10-19 17:30:00,0.4,0.9,7.8,298,29,18.8
...,...,...,...,...,...,...,...,...
49602,camp=06403,2023-01-01 02:00:00,0.3,0.5,3.5,307,43,19.3
49603,camp=06403,2023-01-01 01:30:00,0.3,0.4,2.9,269,36,19.3
49604,camp=06403,2023-01-01 01:00:00,0.3,0.5,3.2,298,50,19.3
49605,camp=06403,2023-01-01 00:30:00,0.2,0.4,4.1,308,35,19.3


In [27]:
num_repeats

4733